# **Interactions in Vega-Altair**

Adapted from Licia He, Elsie Lee, and Eytan Adar and Altair Documentation

# Overview

Today we will learn how to make our Altair charts interactive with tooltips, zooming, panning, etc

**Find the # TODO cells below to practice Altair yourself!**

## Resources
*  [Interactive Charts Documentation](https://altair-viz.github.io/user_guide/interactions/index.html)
*  [Interactive Charts Example Gallery](https://altair-viz.github.io/gallery/index.html#interactive-chartsb)
*  [Multiple Interactions](https://altair-viz.github.io/gallery/multiple_interactions.html)




# Environment Setup

In [ ]:
# Uncomment the below lines if you need to install these packages.
#pip install altair vega_datasets
#pip install pandas

In [ ]:
#package that converts vega-lite charts into static images
!pip install vl-convert-python

In [ ]:
#Import libraries that we need
import pandas as pd
import altair as alt

# Import Data

In [ ]:
from vega_datasets import data

weather = data.seattle_weather()

weather.sample(5)

# Interactions

We'll be talking about interactivity extensively in class but we'd like to introduce you to some of the Altair/Vega-Lite features now. One of the main philosophies in interactivity for infovis is "Overview first, zoom and filter, then details-on-demand." (called Shneiderman's mantra). It basically means that we should give the high level picture and support--through interaction--access to more.  

## Tooltip and Default Interactivity

We'll begin with a familiar interactive element: tooltips. Next, to make exploring complex data easier, we'll add zoom and pan functionality using the  `.interactive()` method.

In [ ]:
chart1 = alt.Chart(weather).mark_circle().encode(
    x=alt.X('temp_max:Q', title='Max Temperature (°C)'),
    y=alt.Y('precipitation:Q', title='Precipitation (mm)'),
    color='weather:N',
    tooltip=[
        alt.Tooltip('date:T', title='Date'),
        alt.Tooltip('temp_max:Q', title='Max Temperature (°C)'),
        alt.Tooltip('precipitation:Q', title='Precipitation (mm)')
    ]
)

chart1

Note that there are other ways to simulate tooltips. For example, this [version](https://altair-viz.github.io/gallery/multiline_tooltip.html) achieves tooltips on the axes by having an invisible layer.

In [ ]:
chart1.interactive() # pan with click-drag, zoom with scroll

## Selection and Conditions

Altair handles interaction through selection. An Altair Selection can

* handle input event
* determine whether or not a given data record lies within the selection
* (re-)configure the visualization based on the selection.

There are two different types of selection in Altair:
1. **Interval selections**: Select ranges of data by clicking and dragging
2. **Point selections**: Select individual data points by clicking or hovering

### Point Selection

To make a single mark selection (hovering, clicking, etc. on a single mark), you need:
1. What kind of selection do we want to make (mouseover/hover, on click, etc.)? Make a single selection instance, called `alt.selection_point()`, which is bound to mouse click by default.
2. Define what happens when the selection happens by adding a [condition](https://altair-viz.github.io/user_guide/generated/api/altair.condition.html#altair.condition)
3. Add this selection (STEP1) to your chart by calling `.add_params()`
4. Include your condition in `encode` or `transform_filter`

We will be using alt.when a lot to specify what happens (step 2). Think of this as a fancy if-then-else. Usually we'll be doing things like this in the encoding:

`color=alt.when(selection).then('weather').otherwise(alt.value('lightgray'))` which roughly means: if the thing is selected, keep it the original color, else set the the color to gray.

In [ ]:
#step 1
selection = alt.selection_point();

#step 2
colorCondition = alt.when(selection).then('weather').otherwise(alt.value('lightgray'))

alt.Chart(weather).mark_circle(
).add_params(
    #step 3
    selection
).encode(
    x=alt.X('temp_max:Q', title='Max Temperature (°C)'),
    y=alt.Y('precipitation:Q', title='Precipitation (mm)'),
    #step4
    color= colorCondition
)

In [ ]:
# TODO: Instead of changing the color of the point
# on selection, change its size

selection = alt.selection_point()

sizeCondition = alt.when(selection).then(alt.SizeValue(200)).otherwise(alt.SizeValue(50))

alt.Chart(weather).mark_circle(
).add_params(
    selection
).encode(
    x=alt.X('temp_max:Q', title='Max Temperature (°C)'),
    y=alt.Y('precipitation:Q', title='Precipitation (mm)'),
    color='weather:N',
    size=sizeCondition
)

By default, `selection_point()` selects all the data points (all have colors, or all have the size of a selected data point). We can override this behavior by setting `empty` parameter to `False`:

In [ ]:
selection = alt.selection_point(empty=False); # now none of the data points is selected by default

colorCondition = alt.when(selection).then('weather').otherwise(alt.value('lightgray'))

alt.Chart(weather).mark_circle(
).add_params(
    selection
).encode(
    x=alt.X('temp_max:Q', title='Max Temperature (°C)'),
    y=alt.Y('precipitation:Q', title='Precipitation (mm)'),
    color= colorCondition
)

In [ ]:
# TODO: set a default empty selection so all points start at their base size

selection = alt.selection_point(empty=False)

sizeCondition = alt.when(selection).then(alt.SizeValue(200)).otherwise(alt.SizeValue(50))

alt.Chart(weather).mark_circle(
).add_params(
    selection
).encode(
    x=alt.X('temp_max:Q', title='Max Temperature (°C)'),
    y=alt.Y('precipitation:Q', title='Precipitation (mm)'),
    color='weather:N',
    size=sizeCondition
)

By default, `selection_point()` responds to clicks. The `on` parameter lets you change that. Let's try to switch it to `mouseover` so hovering over a point is enough to activate the selection

In [ ]:
selection = alt.selection_point(empty=False, on='mouseover'); # passing mouseover

colorCondition = alt.when(selection).then('weather').otherwise(alt.value('lightgray'))

alt.Chart(weather).mark_circle(
).add_params(
    selection
).encode(
    x=alt.X('temp_max:Q', title='Max Temperature (°C)'),
    y=alt.Y('precipitation:Q', title='Precipitation (mm)'),
    color= colorCondition
)

In [ ]:
# TODO: Change the size of the data point, when the user hovers over it

selection = alt.selection_point(empty=False, on='mouseover')

sizeCondition = alt.when(selection).then(alt.SizeValue(200)).otherwise(alt.SizeValue(50))

alt.Chart(weather).mark_circle(
).add_params(
    selection
).encode(
    x=alt.X('temp_max:Q', title='Max Temperature (°C)'),
    y=alt.Y('precipitation:Q', title='Precipitation (mm)'),
    color='weather:N',
    size=sizeCondition
)

The `on` parameter accepts any [event supported](https://vega.github.io/vega/docs/event-streams/#selector) by Vega-Lite, such as `click`, `mouseover`, `dblclick`, and `mouseout`, etc

There are [other parameters](https://altair-viz.github.io/user_guide/generated/api/altair.selection_point.html) that enable us to change the default behavior of `selection_point()`.

### Interval Selection

Switching from point to interval selection requires just one change: replace  `alt.selection_point()` with `alt.selection_interval()`

In [ ]:
#step 1
selection = alt.selection_interval();

#step 2
colorCondition = alt.when(selection).then('weather').otherwise(alt.value('lightgray'))

alt.Chart(weather).mark_circle(
).add_params(
    #step 3
    selection
).encode(
    x=alt.X('temp_max:Q', title='Max Temperature (°C)'),
    y=alt.Y('precipitation:Q', title='Precipitation (mm)'),
    #step4
    color= colorCondition
)

By default, the brush selects points within a 2D rectangle. Setting the `encodings` parameter restricts the selection to a single axis, so brushing along the x-axis selects all points within that time window regardless of their y position. Let's see the example below:

In [ ]:
selection = alt.selection_interval(empty=False, encodings=['x']) # restrict selection to x-axis

colorCondition = alt.when(selection).then('weather').otherwise(alt.value('lightgray'))

alt.Chart(weather).mark_circle(
).add_params(
    selection
).encode(
    x=alt.X('date:T', title='Date'),
    y=alt.Y('temp_max:Q', title='Max Temperature (°C)'),
    color=colorCondition
)

Let's try to provide the user with an initial selection window that they can later change. We can achieve that using the `value` parameter:

In [ ]:
from datetime import datetime

selection = alt.selection_interval(encodings=['x'], value={'x': [datetime(2012, 1, 1), datetime(2012, 7, 1)]})

colorCondition = alt.when(selection).then('weather').otherwise(alt.value('lightgray'))

alt.Chart(weather).mark_circle(
).add_params(
    selection
).encode(
    x=alt.X('date:T', title='Date'),
    y=alt.Y('precipitation:Q', title='Precipitation (mm)'),
    color=colorCondition
)

In [ ]:
# TODO: Add an initial selection window for the summer months (July to September 2012)
# and switch the y-axis to temp_max

from datetime import datetime

selection = alt.selection_interval(encodings=['x'], value={'x': [datetime(2012, 7, 1), datetime(2012, 9, 30)]})

colorCondition = alt.when(selection).then('weather').otherwise(alt.value('lightgray'))

alt.Chart(weather).mark_circle(
).add_params(
    selection
).encode(
    x=alt.X('date:T', title='Date'),
    y=alt.Y('temp_max:Q', title='Max Temperature (°C)'),
    color=colorCondition
)

### Selection and Filtering

Now we can put it all together. The following examples combine compound charts with selections, conditions, and filtering

In [ ]:
# selection: restriction selection to x-axis, set initial window to 6 months
selection = alt.selection_interval(encodings=['x'], value={'x': [datetime(2012, 1, 1), datetime(2012, 7, 1)]})

# color condition
colorCondition = alt.when(selection).then('weather').otherwise(alt.value('lightgray'))

# scatter plot showing maximum temperature over time
scatter = alt.Chart(weather).mark_circle(size=40
  ).add_params(
      selection
  ).encode(
    x=alt.X('date:T', title='Date'),
    y=alt.Y('temp_max:Q', title='Max Temperature (°C)'),
    color=colorCondition
)

# bar chart showing the breakdown of weather types within the selected time window
bars = alt.Chart(weather).mark_bar().encode(
    x=alt.X('count():Q', title='Days'),
    y=alt.Y('weather:N', sort='-x', title=None),
    color='weather:N'
).transform_filter(selection)

scatter | bars


In [ ]:
# selection: restriction selection to x-axis, set initial window to 6 months
selection = alt.selection_interval(encodings=['x'], value={'x': [datetime(2012, 1, 1), datetime(2012, 7, 1)]})

# color condition
colorCondition = alt.when(selection).then('weather').otherwise(alt.value('lightgray'))

# scatter plot showing maximum temperature over time
scatter = alt.Chart(weather).mark_circle(size=40
  ).add_params(
      selection
  ).encode(
    x=alt.X('date:T', title='Date'),
    y=alt.Y('temp_max:Q', title='Max Temperature (°C)'),
    color=colorCondition
).properties(width=350, height=300)

# bar chart showing the breakdown of weather types within the selected time window
bars_count = alt.Chart(weather).mark_bar().encode(
    x=alt.X('count():Q', title='Days'),
    y=alt.Y('weather:N', sort='-x', title=None),
    color='weather:N'
).transform_filter(selection).properties(width=350, height=125)

# TODO: Add bar chart showing mean precipitation per weather type within the selected time window
bars_precip = alt.Chart(weather).mark_bar().encode(
    x=alt.X('mean(precipitation):Q', title='Mean Precipitation (mm)'),
    y=alt.Y('weather:N', sort='-x', title=None),
    color='weather:N'
).transform_filter(selection).properties(width=350, height=125)

scatter | (bars_count & bars_precip)

In [ ]:
# selection: restriction selection to x-axis
selection = alt.selection_interval(encodings=['x'], empty=False)

# opacity condition for the overview chart
opacityCondition = alt.when(selection).then(alt.value(1.0)).otherwise(alt.value(0.1))

overview = alt.Chart(weather).mark_circle(size=20
  ).add_params(
      selection
  ).encode(
    x=alt.X('date:T', title=None),
    y=alt.Y('temp_max:Q', title='Max Temp (°C)', scale=alt.Scale(domain=[-5, 40])),
    color = 'weather:N',
    opacity=opacityCondition
).properties(width=600, height=100)

detail = alt.Chart(weather).mark_circle().encode(
    x=alt.X('date:T', title='Date'),
    y=alt.Y('precipitation:Q', title='Precipitation (mm)'),
    color='weather:N',
    tooltip=['date:T', 'weather:N', 'temp_max:Q', 'precipitation:Q']
).transform_filter(selection).properties(width=600)

overview & detail

In [ ]:
# TODO: build an overview + detail chart for precipitation over time
# overview (top): scatter of date vs precipitation, brushable on x-axis
# use opacity condition to highlight selected points
# detail (bottom): scatter of date vs wind speed, filtered to the brushed window

selection = alt.selection_interval(encodings=['x'], empty=False)

opacityCondition = alt.when(selection).then(alt.value(1.0)).otherwise(alt.value(0.1))

overview = alt.Chart(weather).mark_circle(size=20
  ).add_params(
      selection
  ).encode(
    x=alt.X('date:T', title=None),
    y=alt.Y('precipitation:Q', title='Precipitation (mm)'),
    color='weather:N',
    opacity=opacityCondition
).properties(width=600, height=100)

detail = alt.Chart(weather).mark_circle().encode(
    x=alt.X('date:T', title='Date'),
    y=alt.Y('wind:Q', title='Wind Speed (mph)'),
    color='weather:N',
    tooltip=['date:T', 'weather:N', 'precipitation:Q', 'wind:Q']
).transform_filter(selection).properties(width=600)

overview & detail

## Binding and Widgets

[Bindings](https://altair-viz.github.io/user_guide/interactions/bindings_widgets.html) connect selections to chart elements or UI widgets via the `bind` option. There are three types:

* Data-driven bindings - point and interval selections tied to the data itself, used for highlighting and filtering based on data values (e.g. clicking a [legend](https://altair-viz.github.io/user_guide/interactions/bindings_widgets.html#legend-binding) entry)
* [Widget bindings](https://altair-viz.github.io/user_guide/interactions/bindings_widgets.html#widget-binding) - sliders, dropdowns, and checkboxes that filter or highlight based on absolute values set by the user
* [Scale bindings](https://altair-viz.github.io/user_guide/interactions/bindings_widgets.html#scale-binding) - interval selections tied to a chart's scale, enabling interactions like map zooming

In [ ]:
selection = alt.selection_point(fields=['weather'], bind='legend') # binding legend

#opacity condition
opacityCondition = alt.when(selection).then(alt.value(1.0)).otherwise(alt.value(0.1))

alt.Chart(weather).mark_circle(
  ).add_params(
      selection
  ).encode(
    x=alt.X('date:T', title='Date'),
    y=alt.Y('temp_max:Q', title='Max Temperature (°C)'),
    color='weather:N',
    opacity=opacityCondition
  ).properties(width=600)

### Drop-Down

Let's create a simple drop-down menu. We'll need to do the following:
1.   Obtain a list of options (to populate the drop-down)
2.   Create bind select with `alt.binding_select()` and passing `options` and `name` for our drop down
3.   Create a selection, with `field`, `bind` (we created earlier) and initial `value`
4.   Create a condition

In [ ]:
# first lets create a list of unique weather types
weather_types = sorted(weather['weather'].unique().tolist())

# let's create binding first; it is required to provide options for binding
weather_dropdown = alt.binding_select(options=weather_types, name='Weather Type:')

# now let's apply the binding to selection; start value will be 'drizzle'
selection = alt.selection_point(fields=['weather'], bind=weather_dropdown, value='drizzle')

colorCondition = alt.when(selection).then('weather').otherwise(alt.value('lightgray'))

alt.Chart(weather).mark_circle(
  ).add_params(
      selection
  ).encode(
    x=alt.X('temp_max:Q', title='Max Temperature (°C)'),
    y=alt.Y('precipitation:Q', title='Precipitation (mm)'),
    color = colorCondition,
    tooltip=['date:T', 'weather:N', 'temp_max:Q', 'precipitation:Q']
)


In [ ]:
# TODO: In the chart above, selected points are rendered behind the unselected gray points
# making them difficult to see. Solution: layer a gray background behind colored filtered points.

weather_types = sorted(weather['weather'].unique().tolist())

weather_dropdown = alt.binding_select(options=weather_types, name='Weather Type:')
selection = alt.selection_point(fields=['weather'], bind=weather_dropdown, value='drizzle')

# background layer: all points in gray
background = alt.Chart(weather).mark_circle().encode(
    x=alt.X('temp_max:Q', title='Max Temperature (°C)'),
    y=alt.Y('precipitation:Q', title='Precipitation (mm)'),
    color=alt.value('lightgray'),
    tooltip=['date:T', 'weather:N', 'temp_max:Q', 'precipitation:Q']
)

# foreground layer: only selected points, drawn on top with color
foreground = alt.Chart(weather).mark_circle().encode(
    x=alt.X('temp_max:Q', title='Max Temperature (°C)'),
    y=alt.Y('precipitation:Q', title='Precipitation (mm)'),
    color='weather:N',
    tooltip=['date:T', 'weather:N', 'temp_max:Q', 'precipitation:Q']
).transform_filter(selection)

(background + foreground).add_params(selection)

### Slider

A really useful tool for quantitative variables is the slider. We'll need to do the following:

1. Determine the range of values (to set the `min` and `max` of the slider)
2. Create a range binding with `alt.binding_range()` and passing `min`, `max`, `step`, and `name`
3. Create a selection with `fields`, `bind` (we created earlier), and an initial `value`
4. Create a condition or filter using the selection

In [ ]:
precip_max = float(weather['precipitation'].max())

slider=alt.binding_range(
    min=0,  # min
    max=precip_max,  # max
    step=0.1,              # how many steps to move when the slider adjusts
    name="Min Precipitation (mm): "        # what we call this slider variable
)

#init selection
selector = alt.param(
    bind=slider,        # bind to the slider
    name='cutoff', # it is a variable name that JavaScript will read
    value=0  # start at the max
)

alt.Chart(weather).mark_circle(
).add_params(
    selector
).encode(
    x=alt.X('date:T', title='Date'),
    y=alt.Y('precipitation:Q', title='Precipitation (mm)'),
    color='weather:N',
    tooltip=['date:T', 'weather:N', 'precipitation:Q', 'temp_max:Q', 'wind:Q']
).transform_filter(
    alt.datum.precipitation >= selector
)

In [ ]:
# TODO: build a slider that filters days by maximum wind speed —
# only days below the selected wind threshold should be shown.

wind_max = float(weather['wind'].max())

slider = alt.binding_range(
    min=0,
    max=wind_max,
    step=0.1,
    name="Max Wind Speed: "
)

selector = alt.param(
    bind=slider,
    name='wind_cutoff',
    value=wind_max
)

alt.Chart(weather).mark_circle(
).add_params(
    selector
).encode(
    x=alt.X('date:T', title='Date'),
    y=alt.Y('wind:Q', title='Wind Speed (mph)'),
    color='weather:N',
    tooltip=['date:T', 'weather:N', 'wind:Q', 'temp_max:Q', 'precipitation:Q']
).transform_filter(
    alt.datum.wind <= selector
)

### Checkbox

Let's now try to view how we can use a checkbox to enable interactivity. We'll need to do the following:

1. Decide what the checkbox toggles, it returns `True` when checked and `False` when unchecked
2. Create a checkbox binding with `alt.binding_checkbox()` and passing a `name` label
3. Create a selection with `fields`, `bind` (we created earlier), and an initial `value` (`True` or `False`)
4. Create a condition or filter using the selection to show or hide data based on the checkbox state

In [ ]:
checkbox = alt.binding_checkbox(name='Show rainy days only: ')

selector = alt.param(bind=checkbox, value=False) # Note that we are using alt.param to create selector here

alt.Chart(weather).mark_circle(
).add_params(
    selector
).encode(
    x=alt.X('date:T', title='Date'),
    y=alt.Y('temp_max:Q', title='Max Temperature (°C)'),
    color='weather:N',
    tooltip=['date:T', 'weather:N', 'temp_max:Q', 'precipitation:Q']
).transform_filter(
    # alt.expr.if_(condition, if_true_expr, if_false_expr)
    alt.expr.if_(selector, alt.datum.weather == 'rain', True)
)

Note, we are using `alt.expr.if_` to set the filter condition. You can read the expression the following way:
```
alt.expr.if_(condition, expr_if_true, expr_if_false)
```
For a more detailed list of available expressions and how to use them with interactions, refer to the [documentation](https://altair-viz.github.io/user_guide/interactions/expressions.html).

In [ ]:
# TODO: Build a checkbox that toggles wind data on and off
# when checked, show only days where wind speed exceeds the dataset average

wind_mean = float(weather['wind'].mean())

checkbox = alt.binding_checkbox(name='Show windy days only: ')

selector = alt.param(bind=checkbox, value=False)

alt.Chart(weather).mark_circle(
).add_params(
    selector
).encode(
    x=alt.X('date:T', title='Date'),
    y=alt.Y('wind:Q', title='Wind Speed (mph)'),
    color='weather:N',
    tooltip=['date:T', 'weather:N', 'wind:Q', 'temp_max:Q']
).transform_filter(
    alt.expr.if_(selector, alt.datum.wind > wind_mean, True)
)

### Radio button

Now let's try to use radio buttons. We'll need to do the following:

1. Obtain a list of options (to populate the radio buttons)
2. Create a radio binding with `alt.binding_radio()` and passing `options`, `labels` (optional), and `name`
3. Create a selection with `fields`, `bind` (we created earlier), and an initial `value`
4. Create a condition or filter using the selection

In [ ]:
# extract year to a separate columns
weather['year'] = pd.to_datetime(weather['date']).dt.year

# extract options for radio buttons
years = sorted(weather['year'].unique().tolist())

# create labels for radio buttons
labels = [str(option) + ' ' for option in years]

#we added one more option (none) that cancels the selection and lets us see the data points for all years
radio = alt.binding_radio(options=years+[None], labels = labels+['All'], name='Year: ')

#create selection
selection = alt.selection_point(fields=['year'], bind = radio)

alt.Chart(weather).mark_circle(size=40).add_params(selection).encode(
    x=alt.X('date:T', title='Date'),
    y=alt.Y('temp_max:Q', title='Max Temperature (°C)'),
    color='weather:N',
    tooltip=['date:T', 'weather:N', 'temp_max:Q', 'precipitation:Q']
).transform_filter(selection)

In [ ]:
# TODO: Build a set of radio buttons to filter the chart by weather type
# include an 'All' option that shows the full dataset when selected.

weather_types = sorted(weather['weather'].unique().tolist())
labels = [t + ' ' for t in weather_types]

radio = alt.binding_radio(options=weather_types + [None], labels=labels + ['All'], name='Weather: ')
selection = alt.selection_point(fields=['weather'], bind=radio)

alt.Chart(weather).mark_circle(size=40).add_params(selection).encode(
    x=alt.X('date:T', title='Date'),
    y=alt.Y('temp_max:Q', title='Max Temperature (°C)'),
    color='weather:N',
    tooltip=['date:T', 'weather:N', 'temp_max:Q', 'precipitation:Q']
).transform_filter(selection)

## Multiple Selections and Charts

Let's combine everything we have covered so far into compound interactive charts

In [ ]:
# let's control multiple charts using one widget

# drop down
year_dropdown = alt.binding_select(options=years, name='Year: ')

#selection
selection = alt.selection_point(fields=['year'], bind=year_dropdown)

#base chart -- attach selection and filter
base = alt.Chart(weather).add_params(selection).transform_filter(selection)

# chart 1: temp_max over time
temp_chart = base.mark_line().encode(
    x=alt.X('date:T', title='Date'),
    y=alt.Y('temp_max:Q', title='Max Temperature (°C)')
).properties(width=400, height=125, title='Max Temperature')

# chart 2: precipitation over time
precip_chart = base.mark_bar().encode(
    x=alt.X('date:T', title='Date'),
    y=alt.Y('precipitation:Q', title='Precipitation (mm)')
).properties(width=400, height=125, title='Precipitation')

# chart 3: weather type breakdown
breakdown_chart = base.mark_bar().encode(
    x=alt.X('count():Q', title='Days'),
    y=alt.Y('weather:N', sort='-x', title=None),
    color='weather:N'
).properties(width=400, height=300, title='Weather Type Breakdown')

(temp_chart & precip_chart) | breakdown_chart

In [ ]:
# TODO: Build a dashboard with two widgets that control three charts simultaneously
# a year radio button and a weather type dropdown.

# year radio
year_radio = alt.binding_radio(options=years + [None], labels=[str(y) + ' ' for y in years] + ['All'], name='Year: ')
# year selection
year_selection = alt.selection_point(fields=['year'], bind=year_radio)

# weather dropdown
weather_types = sorted(weather['weather'].unique().tolist())
weather_dropdown = alt.binding_select(options=[None] + weather_types, labels=['All'] + weather_types, name='Weather: ')
# weather selection
weather_selection = alt.selection_point(fields=['weather'], bind=weather_dropdown)

# add both selections with .add_params() and .transform_filter()
base = alt.Chart(weather).add_params(year_selection, weather_selection).transform_filter(year_selection).transform_filter(weather_selection)

# chart1: date vs temp_max; color = weather:N
temp_chart = base.mark_circle().encode(
    x=alt.X('date:T', title='Date'),
    y=alt.Y('temp_max:Q', title='Max Temp (°C)'),
    color='weather:N',
    tooltip=['date:T', 'weather:N', 'temp_max:Q']
).properties(width=250, height=300, title='Temperature')

# chart2: date vs precip; color = weather:N
precip_chart = base.mark_circle().encode(
    x=alt.X('date:T', title='Date'),
    y=alt.Y('precipitation:Q', title='Precipitation (mm)'),
    color='weather:N',
    tooltip=['date:T', 'weather:N', 'precipitation:Q']
).properties(width=250, height=300, title='Precipitation')

# chart3: mean wind speed per weather type
wind_chart = base.mark_bar().encode(
    x=alt.X('mean(wind):Q', title='Mean Wind Speed'),
    y=alt.Y('weather:N', sort='-x', title=None),
    color='weather:N'
).properties(width=250, height=300, title='Wind Speed')

temp_chart | precip_chart | wind_chart

# Exporting Interactive Charts

As a reminder, you can export charts to HMTL directly with the `save()` function:

In [ ]:
precip_max = float(weather['precipitation'].max())

slider=alt.binding_range(
    min=0,  # min
    max=precip_max,  # max
    step=0.1,              # how many steps to move when the slider adjusts
    name="Min Precipitation (mm): "        # what we call this slider variable
)

#init selection
selector = alt.param(
    bind=slider,        # bind to the slider
    name='cutoff', # it is a variable name that JavaScript will read
    value=0  # start at the max
)

chart = alt.Chart(weather).mark_circle(
).add_params(
    selector
).encode(
    x=alt.X('date:T', title='Date'),
    y=alt.Y('precipitation:Q', title='Precipitation (mm)'),
    color='weather:N',
    tooltip=['date:T', 'weather:N', 'precipitation:Q', 'temp_max:Q', 'wind:Q']
).transform_filter(
    alt.datum.precipitation >= selector
)

chart.save("chart.html")

chart

You will now see `chart.html` in the same directory as this notebook.

The above approach embeds the entire dataset in the file. We can avoid this by instead giving altair a URL for the data, instead of passing the data frame directly.

With this approach, the dataset will be retrieved when the page is loaded

In [ ]:
weatherURL = data.seattle_weather.url
print(weatherURL)

In [ ]:
precip_max = float(weather['precipitation'].max())

slider=alt.binding_range(
    min=0,
    max=precip_max,
    step=0.1,
    name="Min Precipitation (mm): "
)

#init selection
selector = alt.param(
    bind=slider,
    name='cutoff',
    value=0
)

#change from DF to URL
chart1 = alt.Chart(weatherURL).mark_circle(
).add_params(
    selector
).encode(
    x=alt.X('date:T', title='Date'),
    y=alt.Y('precipitation:Q', title='Precipitation (mm)'),
    color='weather:N',
    tooltip=['date:T', 'weather:N', 'precipitation:Q', 'temp_max:Q', 'wind:Q']
).transform_filter(
    alt.datum.precipitation >= selector
)
chart1.save("chart1.html")

chart1

If you look at the chart1.html now, it won't work correctly. When data is embedded directly from a DataFrame, Altair pre-converts column types before passing them to Vega-Lite. **However when data loads from a URL at render time, Vega-Lite reads every CSV column as a string, so datum.precipitation becomes "0.5" instead of 0.5, and the numeric comparison that we have in `transform_filter` fails.**

In [ ]:
precip_max = float(weather['precipitation'].max())

slider=alt.binding_range(
    min=0,
    max=precip_max,
    step=0.1,
    name="Min Precipitation (mm): "
)

#init selection
selector = alt.param(
    bind=slider,
    name='cutoff',
    value=0
)

chart2 = alt.Chart(weatherURL).mark_circle(
).add_params(
    selector
).encode(
    x=alt.X('date:T', title='Date'),
    y=alt.Y('precipitation:Q', title='Precipitation (mm)'),
    color='weather:N',
    tooltip=['date:T', 'weather:N', 'precipitation:Q', 'temp_max:Q', 'wind:Q']
).transform_filter(
    # convert Python expression to JS expression
    "toNumber(datum.precipitation) >= cutoff"
)

chart2.save("chart2.html")

chart2

## Putting multiple charts in an HTML file

Let's first review one of the saved .html files:

In [ ]:
html = chart2.to_html()
print(html)

Let's look at whats in the `chart2.html` file.

There are four key sections:

**1. Style**:
```
  <style>
    #vis.vega-embed {
      width: 100%;
      display: flex;
    }

    #vis.vega-embed details,
    #vis.vega-embed details summary {
      position: relative;
    }
  </style>
```

**2. Required scripts**
```
  <script type="text/javascript" src="https://cdn.jsdelivr.net/npm/vega@5"></script>
  <script type="text/javascript" src="https://cdn.jsdelivr.net/npm/vega-lite@5.20.1"></script>
  <script type="text/javascript" src="https://cdn.jsdelivr.net/npm/vega-embed@6"></script>
```

**3. A `div` that holds the chart**
```
<div id="vis"></div>
```

**4. The script with the chart specification**
```
<script>
    (function(vegaEmbed) {
      var spec = {"config": {"view": {"continuousWidth": 300, "continuousHeight": 300}}, "data": {"url": "https://cdn.jsdelivr.net/npm/vega-datasets@v1.29.0/data/seattle-weather.csv"}, "mark": {"type": "circle"}, "encoding": {"color": {"field": "weather", "type": "nominal"}, "tooltip": [{"field": "date", "type": "temporal"}, {"field": "weather", "type": "nominal"}, {"field": "precipitation", "type": "quantitative"}, {"field": "temp_max", "type": "quantitative"}, {"field": "wind", "type": "quantitative"}], "x": {"field": "date", "title": "Date", "type": "temporal"}, "y": {"field": "precipitation", "title": "Precipitation (in)", "type": "quantitative"}}, "params": [{"name": "cutoff", "bind": {"input": "range", "max": 55.9, "min": 0, "name": "Min Precipitation (in): ", "step": 0.1}, "value": 0}], "transform": [{"filter": "toNumber(datum.precipitation) >= cutoff"}], "$schema": "https://vega.github.io/schema/vega-lite/v5.20.1.json"};
      var embedOpt = {"mode": "vega-lite"};

      function showError(el, error){
          el.innerHTML = ('<div style="color:red;">'
                          + '<p>JavaScript Error: ' + error.message + '</p>'
                          + "<p>This usually means there's a typo in your chart specification. "
                          + "See the javascript console for the full traceback.</p>"
                          + '</div>');
          throw error;
      }
      const el = document.getElementById('vis');
      vegaEmbed("#vis", spec, embedOpt)
        .catch(error => showError(el, error));
    })(vegaEmbed);

  </script>
  ```

If we want to put multiple charts in a single html file, we just need to create one div for each chart, and include the corresponding script for each one.

We need to give each chart a distinct ID, and change that in three places: in the div, and in two places in the chart script:
- in the div: `<div id="vis1"></div>`
- in the getElementById function: `const el = document.getElementById('vis1');`
- in the vegaEmbed function: `vegaEmbed("#vis1", spec, embedOpt)`

Let's combine chart1 and chart2 from above into one single html file. If you copy this code to index.html file and open in the browser, you will see charts 1 and 2 from above with a title and some new lines in between

```
<!DOCTYPE html>
<html>
<head>
  <meta charset="UTF-8">
  <style>
    #vis.vega-embed {
      width: 100%;
      display: flex;
    }

    #vis.vega-embed details,
    #vis.vega-embed details summary {
      position: relative;
    }
  </style>
  <script type="text/javascript" src="https://cdn.jsdelivr.net/npm/vega@5"></script>
  <script type="text/javascript" src="https://cdn.jsdelivr.net/npm/vega-lite@5.20.1"></script>
  <script type="text/javascript" src="https://cdn.jsdelivr.net/npm/vega-embed@6"></script>
</head>
<body>
  <div>Chart 1</div>
  <div id="vis1"></div>

  <br/>
  <br/>

  <div>Chart 2</div>
  <div id="vis2"></div>

  <script>
    (function(vegaEmbed) {
      var spec = {"config": {"view": {"continuousWidth": 300, "continuousHeight": 300}}, "data": {"url": "https://cdn.jsdelivr.net/npm/vega-datasets@v1.29.0/data/seattle-weather.csv"}, "mark": {"type": "circle"}, "encoding": {"color": {"field": "weather", "type": "nominal"}, "tooltip": [{"field": "date", "type": "temporal"}, {"field": "weather", "type": "nominal"}, {"field": "precipitation", "type": "quantitative"}, {"field": "temp_max", "type": "quantitative"}, {"field": "wind", "type": "quantitative"}], "x": {"field": "date", "title": "Date", "type": "temporal"}, "y": {"field": "precipitation", "title": "Precipitation (in)", "type": "quantitative"}}, "params": [{"name": "cutoff", "bind": {"input": "range", "max": 55.9, "min": 0, "name": "Min Precipitation (in): ", "step": 0.1}, "value": 0}], "transform": [{"filter": "toNumber(datum.precipitation) >= cutoff"}], "$schema": "https://vega.github.io/schema/vega-lite/v5.20.1.json"};
      var embedOpt = {"mode": "vega-lite"};

      function showError(el, error){
          el.innerHTML = ('<div style="color:red;">'
                          + '<p>JavaScript Error: ' + error.message + '</p>'
                          + "<p>This usually means there's a typo in your chart specification. "
                          + "See the javascript console for the full traceback.</p>"
                          + '</div>');
          throw error;
      }
      const el = document.getElementById('vis1');
      vegaEmbed("#vis1", spec, embedOpt)
        .catch(error => showError(el, error));
    })(vegaEmbed);
  </script>
  <script>
    (function(vegaEmbed) {
      var spec = {"config": {"view": {"continuousWidth": 300, "continuousHeight": 300}}, "data": {"url": "https://cdn.jsdelivr.net/npm/vega-datasets@v1.29.0/data/seattle-weather.csv"}, "mark": {"type": "circle"}, "encoding": {"color": {"field": "weather", "type": "nominal"}, "tooltip": [{"field": "date", "type": "temporal"}, {"field": "weather", "type": "nominal"}, {"field": "precipitation", "type": "quantitative"}, {"field": "temp_max", "type": "quantitative"}, {"field": "wind", "type": "quantitative"}], "x": {"field": "date", "title": "Date", "type": "temporal"}, "y": {"field": "precipitation", "title": "Precipitation (in)", "type": "quantitative"}}, "params": [{"name": "cutoff", "bind": {"input": "range", "max": 55.9, "min": 0, "name": "Min Precipitation (in): ", "step": 0.1}, "value": 0}], "transform": [{"filter": "(datum.precipitation >= cutoff)"}], "$schema": "https://vega.github.io/schema/vega-lite/v5.20.1.json"};
      var embedOpt = {"mode": "vega-lite"};

      function showError(el, error){
          el.innerHTML = ('<div style="color:red;">'
                          + '<p>JavaScript Error: ' + error.message + '</p>'
                          + "<p>This usually means there's a typo in your chart specification. "
                          + "See the javascript console for the full traceback.</p>"
                          + '</div>');
          throw error;
      }
      const el = document.getElementById('vis2');
      vegaEmbed("#vis2", spec, embedOpt)
        .catch(error => showError(el, error));
    })(vegaEmbed);

  </script>
</body>
</html>
```